# 04 - Baseline Classification Models

This notebook performs the first classification experiment using all valid
predictor variables.

The objective is to establish a baseline against which the different
feature-selection strategies will later be compared.

All classifiers are evaluated using the same preprocessing procedure,
cross-validation strategy and evaluation metrics to ensure a fair comparison.

In [1]:
import pandas as pd
import numpy as np

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline

from sklearn.model_selection import (
    StratifiedKFold,
    cross_validate
)

from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

pd.set_option("display.max_columns", None)

## 1. Dataset Preparation

The same preprocessing decisions defined in the previous notebook are applied.

Identifier and constant attributes are removed, `Attrition` is transformed
into a binary target, numerical predictors are standardized and categorical
predictors are one-hot encoded.

The preprocessing operations will be executed inside the classification
pipeline during cross-validation.

In [2]:
DATA_PATH = "../data/raw/WA_Fn-UseC_-HR-Employee-Attrition.csv"

df = pd.read_csv(DATA_PATH)

columns_to_remove = [
    "EmployeeNumber",
    "EmployeeCount",
    "Over18",
    "StandardHours"
]

df_model = df.drop(columns=columns_to_remove)

X = df_model.drop(columns=["Attrition"])

y = df_model["Attrition"].map({
    "No": 0,
    "Yes": 1
})

print("Predictor matrix shape:", X.shape)
print("Target shape:", y.shape)

Predictor matrix shape: (1470, 30)
Target shape: (1470,)


## 2. Preprocessing Definition

Numerical and categorical predictors are identified automatically.

Numerical predictors are standardized, while categorical predictors are
one-hot encoded.

The resulting transformer is included inside each classification pipeline so
that preprocessing is learned independently within every training fold.

In [3]:
numerical_features = X.select_dtypes(
    include=np.number
).columns.tolist()

categorical_features = X.select_dtypes(
    exclude=np.number
).columns.tolist()

print("Numerical predictors:", len(numerical_features))
print("Categorical predictors:", len(categorical_features))

Numerical predictors: 23
Categorical predictors: 7


In [4]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            "num",
            StandardScaler(),
            numerical_features
        ),
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ]
)

## 3. Evaluation Strategy

Model performance is estimated using stratified 10-fold cross-validation.

Stratification preserves approximately the same proportion of attrition and
non-attrition observations in every fold, which is especially important
because the target variable is imbalanced.

For each iteration, nine folds are used for training and the remaining fold is
used for validation.

The procedure is repeated until every fold has been used once as validation
data.

All preprocessing operations are performed exclusively using the training
portion of each fold.

In [5]:
cv = StratifiedKFold(
    n_splits=10,
    shuffle=True,
    random_state=42
)

cv

StratifiedKFold(n_splits=10, random_state=42, shuffle=True)

## 4. Evaluation Metrics

Because the target variable is imbalanced, accuracy alone is not sufficient to
evaluate classification performance.

The following metrics are considered:

- **Accuracy:** proportion of correctly classified observations.
- **Precision:** proportion of predicted attrition cases that actually belong
  to the attrition class.
- **Recall:** proportion of actual attrition cases correctly identified.
- **F1-score:** harmonic mean of precision and recall.
- **Balanced Accuracy:** average recall obtained for both classes.

The positive class corresponds to `Attrition = Yes`.

In [6]:
scoring = {
    "accuracy": "accuracy",
    "precision": "precision",
    "recall": "recall",
    "f1": "f1",
    "balanced_accuracy": "balanced_accuracy"
}

## 5. Classification Algorithms

Several non-probabilistic supervised classification algorithms are evaluated.

The classifiers are initially evaluated using their default or minimally
configured parameters in order to establish a common baseline.

No hyperparameter optimization is performed at this stage. This allows the
effect of the different feature-selection strategies to be analysed under a
consistent experimental setup before introducing additional model tuning.

In [ ]:
models = {
    "K-NN": KNeighborsClassifier(),

    "Decision Tree": DecisionTreeClassifier(
        random_state=42
    ),

    "SVM": SVC(),

    "Neural Network": MLPClassifier(
        random_state=42,
        max_iter=2000
    )
}

## 6. Evaluation Function

A common evaluation function is defined so that every classifier is assessed
using exactly the same preprocessing pipeline, cross-validation strategy and
evaluation metrics.

For every classifier, the function returns the mean and standard deviation of
each metric across the ten validation folds.

In [8]:
def evaluate_model(
    name,
    classifier,
    X,
    y,
    cv,
    scoring
):
    """
    Evaluate a classifier using preprocessing and
    stratified cross-validation.
    """

    pipeline = Pipeline(
        steps=[
            ("preprocessing", preprocessor),
            ("classifier", classifier)
        ]
    )

    scores = cross_validate(
        pipeline,
        X,
        y,
        cv=cv,
        scoring=scoring,
        n_jobs=-1
    )

    result = {
        "Model": name
    }

    for metric in scoring.keys():

        metric_values = scores[f"test_{metric}"]

        result[f"{metric}_mean"] = metric_values.mean()
        result[f"{metric}_std"] = metric_values.std()

    return result

## 7. Model Evaluation

Each classifier is evaluated using the complete set of valid predictor
variables.

The same ten cross-validation folds and evaluation metrics are used for all
models.

In [9]:
results = []

for name, model in models.items():

    print(f"Evaluating {name}...")

    result = evaluate_model(
        name=name,
        classifier=model,
        X=X,
        y=y,
        cv=cv,
        scoring=scoring
    )

    results.append(result)

print("\nEvaluation completed.")

Evaluating K-NN...
Evaluating Decision Tree...
Evaluating SVM...
Evaluating Neural Network...

Evaluation completed.


## 8. Complete Cross-Validation Results

The following table reports both the mean value and the standard deviation of
each evaluation metric across the ten cross-validation folds.

In [10]:
baseline_results = pd.DataFrame(results)

display(
    baseline_results.round(4)
)

,Model,accuracy_mean,accuracy_std,precision_mean,precision_std,recall_mean,recall_std,f1_mean,f1_std,balanced_accuracy_mean,balanced_accuracy_std
0,K-NN,0.8442,0.0162,0.5412,0.2624,0.1350,0.0785,0.2115,0.1101,0.5577,0.0394
1,Decision Tree,0.7925,0.0318,0.3604,0.0939,0.3670,0.1031,0.3620,0.0942,0.6206,0.0564
2,SVM,0.8653,0.0121,0.8363,0.1659,0.2241,0.0551,0.3461,0.0647,0.6064,0.0261
3,Neural Network,0.8490,0.0275,0.5492,0.1137,0.4219,0.0859,0.4738,0.0884,0.6765,0.0475


## 9. Baseline Performance Summary

A simplified table containing the mean performance of each classifier is used
for the main comparison.

Particular attention is given to recall, F1-score and balanced accuracy because
the positive attrition class represents a minority of the observations.

In [11]:
summary_results = baseline_results[
    [
        "Model",
        "accuracy_mean",
        "precision_mean",
        "recall_mean",
        "f1_mean",
        "balanced_accuracy_mean"
    ]
].copy()

summary_results.columns = [
    "Model",
    "Accuracy",
    "Precision",
    "Recall",
    "F1",
    "Balanced Accuracy"
]

summary_results = summary_results.sort_values(
    "F1",
    ascending=False
)

display(
    summary_results.round(4)
)

,Model,Accuracy,Precision,Recall,F1,Balanced Accuracy
3,Neural Network,0.8490,0.5492,0.4219,0.4738,0.6765
1,Decision Tree,0.7925,0.3604,0.3670,0.3620,0.6206
2,SVM,0.8653,0.8363,0.2241,0.3461,0.6064
0,K-NN,0.8442,0.5412,0.1350,0.2115,0.5577


## 10. Export Results

The complete baseline results are stored so that they can later be combined
with the results obtained from the different feature-selection experiments.

In [12]:
baseline_results.to_csv(
    "../results/baseline_results.csv",
    index=False
)

summary_results.to_csv(
    "../results/baseline_summary.csv",
    index=False
)

print("Baseline results saved successfully.")

Baseline results saved successfully.


## 11. Main Observations

The baseline results show substantial differences between overall accuracy and
the ability of the classifiers to identify the minority attrition class.

- The **SVM** achieves the highest overall accuracy (0.8653) and precision
  (0.8363). However, its recall is relatively low (0.2241), indicating that it
  correctly identifies only a limited proportion of employees who actually
  leave the company.

- The **Neural Network** provides the best overall balance between the two
  classes. It obtains the highest recall (0.4219), F1-score (0.4738), and
  balanced accuracy (0.6765) among the evaluated classifiers.

- The **Decision Tree** obtains lower overall accuracy (0.7925), but its recall
  (0.3670) is considerably higher than that of K-NN and SVM. This suggests that
  it is more sensitive to the minority attrition class, although at the cost of
  a larger number of incorrect positive predictions.

- The **K-NN** classifier achieves relatively high accuracy (0.8442), but its
  very low recall (0.1350) and F1-score (0.2115) show that it performs poorly
  when identifying employees belonging to the attrition class.

- These results confirm that accuracy alone is misleading for this dataset.
  Because approximately 84% of the observations belong to the non-attrition
  class, models can obtain high accuracy while still failing to identify a
  large proportion of attrition cases.

- The Neural Network is therefore the strongest baseline according to F1-score
  and balanced accuracy, whereas the SVM provides the highest accuracy and
  precision.

These baseline results will serve as the reference for evaluating whether
univariate, multivariate, and wrapper-based feature selection can improve
predictive performance or achieve comparable performance using fewer
predictors.